<a href="https://colab.research.google.com/github/bugristaya/comp_ling/blob/main/bykova_fine_tuning_hw_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

In [7]:
!pip install transformers datasets evaluate accelerate gradio -q
!pip install huggingface_hub -q

In [8]:
# Для работы с GPU (проверяем наличие)
import torch

print(f"GPU доступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Тип GPU: {torch.cuda.get_device_name(0)}")

GPU доступен: True
Тип GPU: Tesla T4


In [9]:
import numpy as np
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import load_dataset
import evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [10]:
# 1. Загрузка датасета
dataset = load_dataset("fancyzhx/ag_news")
print(f"Датасет загружен. Train: {len(dataset['train'])}, Test: {len(dataset['test'])}")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Датасет загружен. Train: 120000, Test: 7600


In [11]:
# 2. Загрузка модели и токенизатора
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4 # 0 - World, 1 - Sports, 2 - Business, 3 - Sci/Tech
).to(device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
# 3. Подготовка данных и токенизация
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
train_dataset = tokenized_datasets["train"]
eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(5000))

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [13]:
# 4. Настройка обучения
training_args = TrainingArguments(
    output_dir="./ag_news_model", # папка для сохранения модели
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
)

In [14]:
# 5. Метрики
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [15]:
# 6. Обучение (обучение было прервано, т.к. должно было занять около двух часов)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [16]:
# 7. Оценка
eval_results = trainer.evaluate()
print(f"\nEvaluation results: {eval_results}")

Epoch,Training Loss,Validation Loss
0,0.408155,0.278214



Evaluation results: {'eval_loss': 0.2782144844532013}


In [17]:
# 8. Инференс через pipeline
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

test_texts = [
    "Sutton Adds Haas, Cink to Ryder Cup Team MILWAUKEE (Sports Network) - U.S. Ryder Cup captain Hal Sutton finalized his team on Monday when he announced the selections of Jay Haas and Stewart Cink as his captain's picks.",
    "Fears for T N pension after talks Unions representing workers at Turner Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.",
    "Venezuela Prepares for Chavez Recall Vote Supporters and rivals warn of possible fraud; government says Chavez's defeat could produce turmoil in world oil market."
]

for text in test_texts:
    result = classifier(text)[0]
    print(f"Text: {text}\nVerdict: {result['label']}, Score: {result['score']:.4f}\n")

Text: Sutton Adds Haas, Cink to Ryder Cup Team MILWAUKEE (Sports Network) - U.S. Ryder Cup captain Hal Sutton finalized his team on Monday when he announced the selections of Jay Haas and Stewart Cink as his captain's picks.
Verdict: LABEL_1, Score: 0.9910

Text: Fears for T N pension after talks Unions representing workers at Turner Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.
Verdict: LABEL_2, Score: 0.9886

Text: Venezuela Prepares for Chavez Recall Vote Supporters and rivals warn of possible fraud; government says Chavez's defeat could produce turmoil in world oil market.
Verdict: LABEL_0, Score: 0.7140

